# Laboratorio 3 — Redes Neuronales Recurrentes y LSTM

**CC3092 - Deep Learning y Sistemas Inteligentes**

Clasificación binaria de sentimiento sobre el dataset **IMDB Reviews** (positivo / negativo),
comparando tres arquitecturas: un MLP (baseline), una RNN simple (`nn.RNN`) y una LSTM (`nn.LSTM`).

Enlace al repositorio: _pendiente de completar antes de la entrega_.

## 0. Configuración inicial

Importamos las librerías necesarias, fijamos las semillas aleatorias para reproducibilidad y
seleccionamos el dispositivo de cómputo.

**Nota sobre el dispositivo:** en esta máquina (Apple Silicon) está disponible el backend `mps`
de PyTorch, pero se detectó que `nn.RNN` y `nn.LSTM` producen un *crash* nativo en `torch==2.9.1`
sobre `mps` (falla incluso con `PYTORCH_ENABLE_MPS_FALLBACK=1`). Por eso todo el entrenamiento de
este notebook corre sobre **CPU**, usando todos los hilos disponibles. Un *benchmark* rápido mostró
que un paso de entrenamiento de una LSTM (batch=64, seq_len=200, hidden=256, 2 capas) tarda
~0.2s en CPU, lo cual es suficientemente rápido para las 15+ iteraciones que pide el laboratorio.

In [ ]:
import os
import re
import json
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report,
)

from datasets import load_dataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_num_threads(os.cpu_count())
DEVICE = torch.device("cpu")
print("Usando dispositivo:", DEVICE, "| hilos:", torch.get_num_threads())

RESULTS_DIR = "results"
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Dataset

Cargamos el dataset público `imdb` desde HuggingFace `datasets`. Trae una partición `train`
(25,000 reseñas) y una partición `test` (25,000 reseñas), además de una partición `unsupervised`
(50,000 reseñas sin etiqueta) que no usamos en este laboratorio.

In [ ]:
imdb = load_dataset("imdb")
print(imdb)

### 2. Exploración y preparación de los datos

#### 2.1 Observaciones, clases y balance

In [ ]:
train_texts_all = list(imdb["train"]["text"])
train_labels_all = list(imdb["train"]["label"])
test_texts_all = list(imdb["test"]["text"])
test_labels_all = list(imdb["test"]["label"])

print(f"train: {len(train_texts_all)} observaciones")
print(f"test:  {len(test_texts_all)} observaciones")
print("clases:", sorted(set(train_labels_all)), "(0 = negativo, 1 = positivo)")

train_counts = Counter(train_labels_all)
test_counts = Counter(test_labels_all)
print("balance train:", train_counts)
print("balance test:", test_counts)

El dataset tiene 2 clases (`0`=negativo, `1`=positivo). Como se ve arriba, **están perfectamente
balanceadas**: 12,500 reseñas por clase tanto en `train` como en `test`. Esto es importante porque
significa que `accuracy` es una métrica confiable por sí sola (no hay que preocuparse por una clase
mayoritaria dominando el resultado), aunque igual reportamos precision/recall/F1 como pide el
enunciado.

**Nota metodológica:** el split `train` del dataset viene ordenado por clase (primero todas las
negativas, luego todas las positivas), por lo que es indispensable mezclar (`shuffle`) antes de
crear los splits de entrenamiento/validación.

#### 2.2 Longitud de las reseñas

In [ ]:
TOKEN_RE = re.compile(r"[a-z']+")


def tokenize(text: str):
    # Los saltos de línea del HTML original quedan como literal "<br />"; los removemos
    # antes de tokenizar por palabras (regex simple, en minúsculas).
    text = text.replace("<br />", " ")
    return TOKEN_RE.findall(text.lower())


train_tokens_all = [tokenize(t) for t in train_texts_all]
test_tokens_all = [tokenize(t) for t in test_texts_all]

lengths = np.array([len(t) for t in train_tokens_all])
print(f"longitud mínima: {lengths.min()}")
print(f"longitud máxima: {lengths.max()}")
print(f"longitud promedio: {lengths.mean():.1f}")
print(f"longitud mediana: {np.median(lengths):.1f}")
for p in [75, 90, 95, 99]:
    print(f"percentil {p}: {np.percentile(lengths, p):.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(lengths, bins=60, ax=axes[0])
axes[0].axvline(lengths.mean(), color="red", linestyle="--", label=f"media={lengths.mean():.0f}")
axes[0].axvline(np.median(lengths), color="green", linestyle="--", label=f"mediana={np.median(lengths):.0f}")
axes[0].set_title("Distribución de longitud (en tokens) — train")
axes[0].set_xlabel("tokens por reseña")
axes[0].legend()

sns.boxplot(x=lengths, ax=axes[1])
axes[1].set_title("Boxplot de longitud de reseñas")
axes[1].set_xlabel("tokens por reseña")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "length_distribution.png"), dpi=120)
plt.show()

La distribución está sesgada a la derecha: la mayoría de reseñas tiene entre 60 y 300 tokens
(mediana ≈ 170), pero hay una cola larga de reseñas muy extensas (hasta más de 1,000 tokens).

#### 2.3 Tokenización y vocabulario

Usamos una tokenización simple basada en expresiones regulares: minúsculas, se remueven las
etiquetas HTML `<br />` y se extraen secuencias de letras/apóstrofes como tokens
(`[a-z']+`). No usamos un tokenizador de subpalabras (BPE/WordPiece) porque el enunciado pide
trabajar directamente con `nn.Embedding` + capas recurrentes clásicas, no con un modelo
pre-entrenado tipo Transformer.

El vocabulario se construye **únicamente a partir del split de entrenamiento** (para evitar fuga
de información del conjunto de prueba), limitando el tamaño máximo y descartando palabras muy
poco frecuentes.

In [ ]:
VOCAB_MAX_SIZE = 20000
MIN_FREQ = 2
PAD_IDX = 0
UNK_IDX = 1


def build_vocab(token_lists, max_size=VOCAB_MAX_SIZE, min_freq=MIN_FREQ):
    counter = Counter()
    for toks in token_lists:
        counter.update(toks)
    kept = [w for w, c in counter.most_common() if c >= min_freq][: max_size - 2]
    itos = ["<pad>", "<unk>"] + kept
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos, counter


# El vocabulario se construye más abajo, sobre el split de train ya separado de validación,
# para no filtrar información del split de validación tampoco.

**Manejo de out-of-vocabulary (OOV):** cualquier palabra que no esté en el vocabulario (porque no
alcanzó `min_freq=2` ocurrencias en train, o porque solo aparece en validación/test) se mapea al
token especial `<unk>` (índice 1). El token `<pad>` (índice 0) se usa para rellenar secuencias
más cortas que la longitud máxima del batch.

#### 2.4 Padding y truncamiento

Las reseñas tienen longitud variable, pero para procesarlas en batches con operaciones
matriciales (tanto en el MLP como en la RNN/LSTM) todas las secuencias de un mismo batch deben
tener la misma longitud. Por eso:

- **Padding**: las secuencias más cortas que la máxima del batch se rellenan con `<pad>` (índice 0)
  hasta igualar la longitud. Usamos `nn.utils.rnn.pad_sequence`.
- **Truncamiento**: fijamos una longitud máxima `MAX_LEN` y cortamos las reseñas más largas, para
  acotar el costo computacional y de memoria (sin esto, una sola reseña de 1,500 tokens dominaría
  el costo del batch completo).
- En la RNN/LSTM usamos además `pack_padded_sequence`, que le indica a PyTorch la longitud real de
  cada secuencia para que el cómputo recurrente no pierda tiempo (ni aprenda) sobre las posiciones
  de padding.

Con base en la distribución de longitudes (mediana ≈ 170, percentil 90 ≈ 440), elegimos
**`MAX_LEN = 256`** para la búsqueda principal de hiperparámetros: cubre la mayoría de reseñas
casi completas sin disparar el costo computacional por la cola larga de reseñas extremadamente
extensas. En la sección 4.1 exploramos explícitamente el efecto de usar `MAX_LEN` mucho menor
(50) y mucho mayor (450).

#### 2.5 Ejemplos del dataset

In [ ]:
MAX_LEN = 256

rng = random.Random(SEED)
example_idx = rng.sample(range(len(train_texts_all)), 5)
examples = []
for i in example_idx:
    toks = train_tokens_all[i]
    examples.append({
        "texto (primeros 200 caracteres)": train_texts_all[i][:200].replace("\n", " ") + "...",
        "etiqueta": "positivo" if train_labels_all[i] == 1 else "negativo",
        "longitud (tokens)": len(toks),
    })
pd.DataFrame(examples)

#### 2.6 División en entrenamiento, validación y test

El dataset original solo trae `train`/`test`. Creamos el split de **validación** separando un
10% del split `train` (de forma estratificada por clase, y mezclando primero porque el split
viene ordenado por etiqueta). El split `test` original se deja intacto y solo se usa **una vez**,
al final, para la evaluación final de los tres modelos.

In [ ]:
idx_train_full, idx_val = train_test_split(
    np.arange(len(train_texts_all)),
    test_size=0.10,
    stratify=train_labels_all,
    random_state=SEED,
)

train_tokens = [train_tokens_all[i] for i in idx_train_full]
train_labels = [train_labels_all[i] for i in idx_train_full]
val_tokens = [train_tokens_all[i] for i in idx_val]
val_labels = [train_labels_all[i] for i in idx_val]
test_tokens = test_tokens_all
test_labels = test_labels_all

print(f"train: {len(train_tokens)} | val: {len(val_tokens)} | test: {len(test_tokens)}")
print("balance train:", Counter(train_labels))
print("balance val:", Counter(val_labels))

stoi, itos, train_counter = build_vocab(train_tokens)
VOCAB_SIZE = len(stoi)
print(f"tamaño de vocabulario: {VOCAB_SIZE}")

## 3. Investigación: capas de PyTorch para RNN y LSTM

| Capa / utilidad | Propósito | Parámetros más relevantes |
|---|---|---|
| `nn.Embedding` | Tabla de *lookup* entrenable que mapea índices de tokens (enteros) a vectores densos de dimensión fija. Reemplaza a un one-hot + matriz densa, mucho más eficiente en memoria. | `num_embeddings` (tamaño del vocabulario), `embedding_dim`, `padding_idx` (el índice cuyo vector se fija en cero y no recibe gradiente, típicamente el de `<pad>`). |
| `nn.RNN` | Red recurrente "vanilla": en cada paso de tiempo combina la entrada actual con el estado oculto anterior mediante una transformación lineal + no linealidad (`tanh` o `relu`), produciendo un nuevo estado oculto. Sufre de vanishing/exploding gradient en secuencias largas (ver más abajo). | `input_size`, `hidden_size`, `num_layers` (apilar varias capas recurrentes), `nonlinearity` (`tanh`/`relu`), `batch_first`, `dropout` (dropout entre capas si `num_layers>1`), `bidirectional`. |
| `nn.LSTM` | Variante recurrente con memoria de largo plazo explícita (estado de celda `c_t`) y tres compuertas (forget, input, output) que controlan qué información se olvida, se agrega y se expone como salida. Mitiga el vanishing gradient de la RNN simple (ver sección de conceptos). | Mismos parámetros que `nn.RNN` (`input_size`, `hidden_size`, `num_layers`, `batch_first`, `dropout`, `bidirectional`), sin `nonlinearity` (las compuertas usan sigmoid/tanh internamente y no es configurable). |
| `nn.utils.rnn.pad_sequence` | Recibe una lista de tensores 1D (secuencias) de longitud variable y las apila en un único tensor rellenando con un valor de padding (`padding_value`, típicamente 0) hasta la longitud de la secuencia más larga del batch. | `sequences`, `batch_first`, `padding_value`. |
| `nn.utils.rnn.pack_padded_sequence` | Comprime un tensor ya paddeado (junto con las longitudes reales de cada secuencia) en un `PackedSequence`, una representación que le permite a `nn.RNN`/`nn.LSTM` procesar solo los pasos de tiempo reales de cada secuencia (evitando cómputo y gradiente sobre el padding), lo cual además es más eficiente. | `input`, `lengths`, `batch_first`, `enforce_sorted` (si `False`, no exige que el batch venga ordenado por longitud descendente). |
| `nn.utils.rnn.pad_packed_sequence` | Operación inversa: convierte un `PackedSequence` (por ejemplo, la salida completa `output` de una `nn.RNN`/`nn.LSTM`) de vuelta a un tensor denso con padding, útil si se necesita la salida en cada paso de tiempo (arquitecturas many-to-many). En nuestra arquitectura many-to-one no la necesitamos porque solo usamos el último estado oculto `h_n`, pero se investiga por completitud. | `sequence`, `batch_first`, `padding_value`, `total_length`. |
| `torch.nn.utils.clip_grad_norm_` | Recorta ("clippea") la norma global de los gradientes de todos los parámetros de un modelo a un valor máximo (`max_norm`) *in-place*, justo antes de `optimizer.step()`. Se usa para controlar el problema de **exploding gradient**, muy común al entrenar RNNs sobre secuencias largas. | `parameters`, `max_norm`, `norm_type` (por defecto norma L2). |
| `nn.Dropout` en capas recurrentes | Apaga aleatoriamente una fracción de activaciones durante entrenamiento para regularizar. En `nn.RNN`/`nn.LSTM`, el parámetro `dropout` del constructor aplica dropout **entre capas apiladas** (solo tiene efecto si `num_layers > 1`); para regularizar también con una sola capa, se agrega explícitamente una `nn.Dropout` sobre el estado oculto final antes de la capa de clasificación (lo que hacemos en este notebook). | `p` (probabilidad de apagar una unidad). |

### Conceptos adicionales

**Hidden state vs. cell state (LSTM).** El *hidden state* `h_t` es la salida visible de la celda
en el paso `t` (lo que se pasa a la siguiente capa o se usa para predecir); es análogo al estado
oculto de una RNN simple. El *cell state* `c_t` es una "cinta transportadora" interna de memoria de
largo plazo que fluye casi sin transformaciones no lineales entre pasos de tiempo (solo
multiplicaciones/sumas moduladas por las compuertas), lo que le permite preservar información por
muchos pasos sin que el gradiente se desvanezca. `h_t` se calcula a partir de `c_t` filtrado por la
compuerta de salida.

**Vanishing / exploding gradient.** Al hacer *backpropagation through time*, el gradiente respecto
a los parámetros de pasos de tiempo lejanos se calcula como un producto de muchos jacobianos (uno
por paso de tiempo). Si las normas de esos jacobianos son sistemáticamente menores a 1, el
producto tiende a 0 exponencialmente (*vanishing gradient*): la red deja de poder aprender
dependencias de largo plazo. Si son mayores a 1, el producto crece exponencialmente (*exploding
gradient*), causando actualizaciones de peso inestables o `NaN`. El efecto es multiplicativo en
la cantidad de pasos de tiempo, por lo que empeora exponencialmente con secuencias más largas.

**Cómo las compuertas de la LSTM mitigan el vanishing gradient.** La compuerta de *forget*
controla, paso a paso, cuánta memoria previa se conserva; cuando se aprende a mantenerla cercana a
1, el camino del gradiente a través de `c_t` se vuelve casi aditivo (suma de contribuciones en vez
de un producto largo de jacobianos con `tanh`/`sigmoid`), permitiendo que el gradiente fluya hacia
atrás muchos pasos sin desvanecerse tan agresivamente. Las compuertas de *input* y *output*
controlan qué nueva información entra a la memoria y qué parte de la memoria se expone como salida
en cada paso, dándole a la red control explícito sobre qué recordar/olvidar/exponer, en vez de
reescribir todo el estado oculto en cada paso como hace una RNN simple.

## 4. Construcción y entrenamiento de las arquitecturas

Definimos las tres arquitecturas y las utilidades de entrenamiento/evaluación compartidas.

- **MLP (baseline)**: usa como representación de la reseña el **promedio de los embeddings**
  de sus palabras (ignorando el padding mediante una máscara), y pasa ese vector por una o más
  capas densas.
- **RNN / LSTM (many-to-one)**: alimentan la secuencia completa de embeddings, palabra por
  palabra, a `nn.RNN`/`nn.LSTM` (usando `pack_padded_sequence` para ignorar el padding), y usan el
  **último estado oculto** (`h_n` de la última capa) como representación de la reseña para
  clasificar.

Todas comparten la misma capa `nn.Embedding` de entrada (entrenada desde cero, no
pre-entrenada) y la misma cabeza de clasificación lineal final (2 clases).

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, encoded, labels):
        self.encoded = encoded
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.encoded[idx], dtype=torch.long), self.labels[idx]


def encode(tokens, stoi, max_len):
    ids = [stoi.get(t, UNK_IDX) for t in tokens[:max_len]]
    if len(ids) == 0:
        ids = [UNK_IDX]
    return ids


def collate_batch(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded = pad_sequence(list(seqs), batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels


def make_loaders(max_len, batch_size=64):
    train_enc = [encode(t, stoi, max_len) for t in train_tokens]
    val_enc = [encode(t, stoi, max_len) for t in val_tokens]
    test_enc = [encode(t, stoi, max_len) for t in test_tokens]

    train_ds = IMDBDataset(train_enc, train_labels)
    val_ds = IMDBDataset(val_enc, val_labels)
    test_ds = IMDBDataset(test_enc, test_labels)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
    return train_dl, val_dl, test_dl


# Loaders para la búsqueda principal de hiperparámetros (MAX_LEN = 256).
train_dl, val_dl, test_dl = make_loaders(MAX_LEN)
print("batches -> train:", len(train_dl), "val:", len(val_dl), "test:", len(test_dl))

In [ ]:
class MLPClassifier(nn.Module):
    """Baseline: promedio de embeddings (bag-of-embeddings) + MLP."""

    def __init__(self, vocab_size, emb_dim, hidden_dims, dropout=0.3, n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        layers = []
        in_dim = emb_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, n_classes))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x, lengths):
        emb = self.embedding(x)  # (B, T, E)
        mask = (x != PAD_IDX).unsqueeze(-1).float()
        avg = (emb * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
        return self.mlp(avg)


class RecurrentClassifier(nn.Module):
    """RNN o LSTM many-to-one: usa el último hidden state para clasificar."""

    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers=1, dropout=0.0,
                 n_classes=2, cell="rnn"):
        super().__init__()
        assert cell in ("rnn", "lstm")
        self.cell = cell
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        rnn_cls = nn.LSTM if cell == "lstm" else nn.RNN
        kwargs = dict(
            input_size=emb_dim, hidden_size=hidden_size, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
        )
        if cell == "rnn":
            kwargs["nonlinearity"] = "tanh"
        self.rnn = rnn_cls(**kwargs)
        # Dropout explícito sobre el último hidden state, para regularizar incluso con
        # num_layers=1 (el dropout interno de nn.RNN/nn.LSTM solo actúa entre capas apiladas).
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        if self.cell == "lstm":
            _, (h_n, _) = self.rnn(packed)
        else:
            _, h_n = self.rnn(packed)
        last_layer_hidden = h_n[-1]  # (B, hidden_size), última capa apilada
        return self.fc(self.dropout(last_layer_hidden))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def run_epoch(model, loader, optimizer=None, clip=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(is_train):
        for x, lengths, y in loader:
            logits = model(x, lengths)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                if clip is not None:
                    nn.utils.clip_grad_norm_(model.parameters(), clip)
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(logits.argmax(dim=1).detach())
            all_labels.append(y.detach())
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    avg_loss = total_loss / len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    metrics = {"accuracy": acc, "precision": p, "recall": r, "f1": f1}
    return avg_loss, metrics


def train_model(model, train_dl, val_dl, epochs, lr, weight_decay=0.0, clip=None, verbose=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "val_metrics": []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        train_loss, _ = run_epoch(model, train_dl, optimizer, clip=clip)
        val_loss, val_metrics = run_epoch(model, val_dl, optimizer=None)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_metrics"].append(val_metrics)
        if verbose:
            print(f"  epoch {epoch}/{epochs} | train_loss={train_loss:.4f} "
                  f"val_loss={val_loss:.4f} val_f1={val_metrics['f1']:.4f}")
    elapsed = time.time() - t0
    return history, elapsed


def run_config(name, arch, build_fn, train_dl, val_dl, epochs, lr, weight_decay=0.0, clip=None,
                extra=None):
    torch.manual_seed(SEED)
    model = build_fn()
    n_params = count_params(model)
    print(f"[{arch}] {name} -> params={n_params:,}")
    history, elapsed = train_model(model, train_dl, val_dl, epochs, lr, weight_decay, clip, verbose=True)
    best_epoch = int(np.argmax([m["f1"] for m in history["val_metrics"]]))
    result = {
        "arch": arch,
        "name": name,
        "hyperparams": {
            "lr": lr, "weight_decay": weight_decay, "clip": clip, "epochs": epochs,
            **(extra or {}),
        },
        "n_params": n_params,
        "history": history,
        "best_epoch": best_epoch + 1,
        "best_val_metrics": history["val_metrics"][best_epoch],
        "final_val_metrics": history["val_metrics"][-1],
        "train_time_sec": elapsed,
    }
    print(f"  -> mejor epoch={best_epoch + 1} val_f1={result['best_val_metrics']['f1']:.4f} "
          f"tiempo={elapsed:.1f}s")
    return model, result

## 5. Iteraciones de hiperparámetros

Para cada arquitectura probamos 5 configuraciones (15 en total), variando un hiperparámetro a la
vez respecto a una configuración base, para poder atribuir el efecto de cada cambio con más
claridad en la sección de discusión. Todas las iteraciones usan `EPOCHS=6`, `BATCH_SIZE=64` y
`MAX_LEN=256`.

### 5.1 MLP

In [ ]:
EPOCHS = 6
EMB_DIM = 100

mlp_configs = [
    {"name": "baseline (hidden=[128], dropout=0.3)", "hidden_dims": [128], "dropout": 0.3, "lr": 1e-3, "weight_decay": 0.0},
    {"name": "mas profundo (hidden=[256,64])", "hidden_dims": [256, 64], "dropout": 0.3, "lr": 1e-3, "weight_decay": 0.0},
    {"name": "sin dropout", "hidden_dims": [128], "dropout": 0.0, "lr": 1e-3, "weight_decay": 0.0},
    {"name": "lr alto (1e-2)", "hidden_dims": [128], "dropout": 0.3, "lr": 1e-2, "weight_decay": 0.0},
    {"name": "weight decay (1e-4)", "hidden_dims": [128], "dropout": 0.3, "lr": 1e-3, "weight_decay": 1e-4},
]

mlp_results = []
mlp_models = {}
for cfg in mlp_configs:
    def build(cfg=cfg):
        return MLPClassifier(VOCAB_SIZE, EMB_DIM, cfg["hidden_dims"], dropout=cfg["dropout"])
    model, res = run_config(
        cfg["name"], "MLP", build, train_dl, val_dl, EPOCHS,
        lr=cfg["lr"], weight_decay=cfg["weight_decay"],
        extra={"hidden_dims": cfg["hidden_dims"], "dropout": cfg["dropout"]},
    )
    mlp_results.append(res)
    mlp_models[cfg["name"]] = model

### 5.2 RNN simple (`nn.RNN`)

In [ ]:
rnn_configs = [
    {"name": "hidden=64", "hidden_size": 64, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "clip": 5.0},
    {"name": "baseline (hidden=128)", "hidden_size": 128, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "clip": 5.0},
    {"name": "sin gradient clipping", "hidden_size": 128, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "clip": None},
    {"name": "sin dropout", "hidden_size": 128, "num_layers": 1, "dropout": 0.0, "lr": 1e-3, "clip": 5.0},
    {"name": "2 capas apiladas", "hidden_size": 128, "num_layers": 2, "dropout": 0.3, "lr": 1e-3, "clip": 5.0},
]

rnn_results = []
rnn_models = {}
for cfg in rnn_configs:
    def build(cfg=cfg):
        return RecurrentClassifier(
            VOCAB_SIZE, EMB_DIM, cfg["hidden_size"], num_layers=cfg["num_layers"],
            dropout=cfg["dropout"], cell="rnn",
        )
    model, res = run_config(
        cfg["name"], "RNN", build, train_dl, val_dl, EPOCHS,
        lr=cfg["lr"], clip=cfg["clip"],
        extra={"hidden_size": cfg["hidden_size"], "num_layers": cfg["num_layers"], "dropout": cfg["dropout"]},
    )
    rnn_results.append(res)
    rnn_models[cfg["name"]] = model

### 5.3 LSTM (`nn.LSTM`)\n\nMisma rejilla de configuraciones que la RNN, para poder comparar directamente el efecto de las compuertas de memoria bajo condiciones idénticas.

In [ ]:
lstm_configs = [
    {"name": "hidden=64", "hidden_size": 64, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "clip": 5.0},
    {"name": "baseline (hidden=128)", "hidden_size": 128, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "clip": 5.0},
    {"name": "sin gradient clipping", "hidden_size": 128, "num_layers": 1, "dropout": 0.3, "lr": 1e-3, "clip": None},
    {"name": "sin dropout", "hidden_size": 128, "num_layers": 1, "dropout": 0.0, "lr": 1e-3, "clip": 5.0},
    {"name": "2 capas apiladas", "hidden_size": 128, "num_layers": 2, "dropout": 0.3, "lr": 1e-3, "clip": 5.0},
]

lstm_results = []
lstm_models = {}
for cfg in lstm_configs:
    def build(cfg=cfg):
        return RecurrentClassifier(
            VOCAB_SIZE, EMB_DIM, cfg["hidden_size"], num_layers=cfg["num_layers"],
            dropout=cfg["dropout"], cell="lstm",
        )
    model, res = run_config(
        cfg["name"], "LSTM", build, train_dl, val_dl, EPOCHS,
        lr=cfg["lr"], clip=cfg["clip"],
        extra={"hidden_size": cfg["hidden_size"], "num_layers": cfg["num_layers"], "dropout": cfg["dropout"]},
    )
    lstm_results.append(res)
    lstm_models[cfg["name"]] = model

### 5.4 Resumen de iteraciones y curvas de pérdida

Tabla con hiperparámetros, mejores métricas de validación y número de parámetros de las 15
iteraciones, seguida de las curvas de pérdida de entrenamiento/validación (todas las
iteraciones, superando el mínimo de 3 por arquitectura pedido).

In [ ]:
all_results = mlp_results + rnn_results + lstm_results

summary_rows = []
for r in all_results:
    row = {
        "arquitectura": r["arch"],
        "iteración": r["name"],
        "n_params": r["n_params"],
        "mejor_epoch": r["best_epoch"],
        "val_accuracy": r["best_val_metrics"]["accuracy"],
        "val_precision": r["best_val_metrics"]["precision"],
        "val_recall": r["best_val_metrics"]["recall"],
        "val_f1": r["best_val_metrics"]["f1"],
        "tiempo_s": r["train_time_sec"],
    }
    row.update({f"hp_{k}": v for k, v in r["hyperparams"].items()})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df_display = summary_df[[
    "arquitectura", "iteración", "n_params", "mejor_epoch",
    "val_accuracy", "val_precision", "val_recall", "val_f1", "tiempo_s",
]].round(4)
summary_df_display

In [ ]:
def plot_loss_curves(results, arch_name):
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.2), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, r in zip(axes, results):
        epochs_range = range(1, len(r["history"]["train_loss"]) + 1)
        ax.plot(epochs_range, r["history"]["train_loss"], label="train")
        ax.plot(epochs_range, r["history"]["val_loss"], label="val")
        ax.set_title(r["name"], fontsize=9)
        ax.set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].legend()
    fig.suptitle(f"Curvas de pérdida — {arch_name} (5 iteraciones)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"loss_curves_{arch_name.lower()}.png"), dpi=120)
    plt.show()


plot_loss_curves(mlp_results, "MLP")
plot_loss_curves(rnn_results, "RNN")
plot_loss_curves(lstm_results, "LSTM")

### 5.5 Selección de la mejor configuración por arquitectura

Elegimos, para cada arquitectura, la iteración con mayor F1 (macro) de validación en su mejor
epoch.

In [ ]:
def pick_best(results):
    return max(results, key=lambda r: r["best_val_metrics"]["f1"])


best_mlp_res = pick_best(mlp_results)
best_rnn_res = pick_best(rnn_results)
best_lstm_res = pick_best(lstm_results)

best_mlp_model = mlp_models[best_mlp_res["name"]]
best_rnn_model = rnn_models[best_rnn_res["name"]]
best_lstm_model = lstm_models[best_lstm_res["name"]]

for arch, res in [("MLP", best_mlp_res), ("RNN", best_rnn_res), ("LSTM", best_lstm_res)]:
    print(f"Mejor {arch}: {res['name']} | val_f1={res['best_val_metrics']['f1']:.4f} "
          f"| val_acc={res['best_val_metrics']['accuracy']:.4f} | params={res['n_params']:,}")

## 6. Evaluación final sobre test

Evaluamos **una única vez** las tres mejores configuraciones sobre el conjunto de test (25,000
reseñas, nunca antes visto durante el entrenamiento ni la selección de hiperparámetros), y
generamos la matriz de confusión de cada una.

In [ ]:
def evaluate_on_test(model, test_dl):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, lengths, y in test_dl:
            logits = model(x, lengths)
            all_preds.append(logits.argmax(dim=1))
            all_labels.append(y)
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}, cm, all_preds, all_labels


final_models = {"MLP": best_mlp_model, "RNN": best_rnn_model, "LSTM": best_lstm_model}
final_test_results = {}
final_preds = {}

for arch, model in final_models.items():
    metrics, cm, preds, labels = evaluate_on_test(model, test_dl)
    final_test_results[arch] = {"metrics": metrics, "confusion_matrix": cm.tolist()}
    final_preds[arch] = preds
    print(f"{arch} -> test_acc={metrics['accuracy']:.4f} test_f1={metrics['f1']:.4f} "
          f"precision={metrics['precision']:.4f} recall={metrics['recall']:.4f}")

pd.DataFrame({arch: r["metrics"] for arch, r in final_test_results.items()}).T.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (arch, r) in zip(axes, final_test_results.items()):
    cm = np.array(r["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
    ax.set_title(f"Matriz de confusión — {arch}")
    ax.set_xlabel("predicho")
    ax.set_ylabel("real")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "confusion_matrices.png"), dpi=120)
plt.show()

### Ejemplos mal clasificados

Inspeccionamos algunos ejemplos de test donde el mejor modelo (por F1) se equivocó, para
alimentar la discusión sobre qué tipo de reseñas resultan más difíciles.

In [ ]:
best_arch_by_f1 = max(final_test_results, key=lambda a: final_test_results[a]["metrics"]["f1"])
print("Mejor arquitectura en test:", best_arch_by_f1)

errors_idx = np.where(final_preds[best_arch_by_f1] != np.array(test_labels))[0]
rng2 = random.Random(SEED)
sample_errors = rng2.sample(list(errors_idx), min(6, len(errors_idx)))

misclassified_examples = []
for i in sample_errors:
    misclassified_examples.append({
        "texto (primeros 300 caracteres)": test_texts_all[i][:300].replace("\n", " "),
        "etiqueta real": "positivo" if test_labels[i] == 1 else "negativo",
        "predicción": "positivo" if final_preds[best_arch_by_f1][i] == 1 else "negativo",
        "longitud (tokens)": len(test_tokens[i]),
    })
misclassified_df = pd.DataFrame(misclassified_examples)
misclassified_df

## 4.1 Experimento adicional: efecto de la longitud de secuencia

Usando las mejores configuraciones encontradas para la RNN y la LSTM, las volvemos a entrenar
(desde cero, mismos hiperparámetros, misma cantidad de epochs) sobre dos versiones del dataset
con longitud máxima de secuencia distinta: **`MAX_LEN=50`** (recorta agresivamente casi toda
reseña) y **`MAX_LEN=450`** (conserva prácticamente toda la reseña para la gran mayoría de
casos, según el percentil 95 calculado en la sección 2.2).

In [ ]:
SEQ_LENS = [50, 450]
seqlen_results = {}

for seq_len in SEQ_LENS:
    tr_dl, va_dl, te_dl = make_loaders(seq_len)
    for arch, best_res in [("RNN", best_rnn_res), ("LSTM", best_lstm_res)]:
        hp = best_res["hyperparams"]

        def build(hp=hp, arch=arch):
            return RecurrentClassifier(
                VOCAB_SIZE, EMB_DIM, hp["hidden_size"], num_layers=hp["num_layers"],
                dropout=hp["dropout"], cell=arch.lower(),
            )

        name = f"{arch} (mejor config) @ max_len={seq_len}"
        model, res = run_config(
            name, arch, build, tr_dl, va_dl, EPOCHS,
            lr=hp["lr"], clip=hp["clip"], extra={"max_len": seq_len, **{k: v for k, v in hp.items() if k not in ("lr", "clip", "epochs")}},
        )
        test_metrics, cm, preds, labels = evaluate_on_test(model, te_dl)
        seqlen_results[(arch, seq_len)] = {
            "train_result": res,
            "test_metrics": test_metrics,
            "confusion_matrix": cm.tolist(),
        }
        print(f"  test -> acc={test_metrics['accuracy']:.4f} f1={test_metrics['f1']:.4f}")

In [ ]:
seqlen_rows = []
for (arch, seq_len), r in seqlen_results.items():
    seqlen_rows.append({
        "arquitectura": arch,
        "max_len": seq_len,
        "val_f1 (mejor epoch)": r["train_result"]["best_val_metrics"]["f1"],
        "test_accuracy": r["test_metrics"]["accuracy"],
        "test_precision": r["test_metrics"]["precision"],
        "test_recall": r["test_metrics"]["recall"],
        "test_f1": r["test_metrics"]["f1"],
        "tiempo_s": r["train_result"]["train_time_sec"],
    })
seqlen_df = pd.DataFrame(seqlen_rows).sort_values(["arquitectura", "max_len"]).round(4)
seqlen_df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for arch in ["RNN", "LSTM"]:
    sub = seqlen_df[seqlen_df["arquitectura"] == arch].sort_values("max_len")
    ax.plot(sub["max_len"], sub["test_f1"], marker="o", label=arch)
ax.set_xlabel("longitud máxima de secuencia (max_len)")
ax.set_ylabel("F1 (macro) en test")
ax.set_title("Efecto de la longitud de secuencia en el desempeño")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "seqlen_effect.png"), dpi=120)
plt.show()

## 5. Comparación de arquitecturas

Comparamos las tres mejores configuraciones (evaluadas sobre el mismo conjunto de test) según
número de parámetros entrenables, desempeño y tiempo de entrenamiento.

In [ ]:
comparison_rows = []
for arch, best_res in [("MLP", best_mlp_res), ("RNN", best_rnn_res), ("LSTM", best_lstm_res)]:
    m = final_test_results[arch]["metrics"]
    comparison_rows.append({
        "arquitectura": arch,
        "configuración": best_res["name"],
        "n_params": best_res["n_params"],
        "test_accuracy": m["accuracy"],
        "test_precision": m["precision"],
        "test_recall": m["recall"],
        "test_f1": m["f1"],
        "tiempo_entrenamiento_s": best_res["train_time_sec"],
        "f1_por_1000_params": m["f1"] / (best_res["n_params"] / 1000),
    })
comparison_df = pd.DataFrame(comparison_rows).round(4)
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(comparison_df["arquitectura"], comparison_df["n_params"], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Parámetros entrenables por arquitectura")
axes[0].set_ylabel("n_params")

axes[1].bar(comparison_df["arquitectura"], comparison_df["test_f1"], color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("F1 (macro) en test por arquitectura")
axes[1].set_ylabel("F1")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "comparison_params_f1.png"), dpi=120)
plt.show()

El experimento de la sección 4.1 (efecto de la longitud de secuencia) para RNN y LSTM se resume
en la tabla `seqlen_df` de la sección anterior y en la figura `seqlen_effect.png`.

## 6. Discusión y análisis

Esta sección se completa automáticamente a partir de los resultados obtenidos arriba (celda de
código siguiente), y luego se interpreta en las celdas de markdown posteriores.

In [ ]:
def hp_diff(base_res, other_res):
    base_hp, other_hp = base_res["hyperparams"], other_res["hyperparams"]
    diffs = {k: (base_hp.get(k), other_hp.get(k)) for k in other_hp if base_hp.get(k) != other_hp.get(k)}
    return diffs


def best_and_worst_change(results, baseline_name_substr="baseline"):
    baseline = next(r for r in results if baseline_name_substr in r["name"] or r is results[1])
    others = [r for r in results if r is not baseline]
    deltas = [(r, r["best_val_metrics"]["f1"] - baseline["best_val_metrics"]["f1"]) for r in others]
    best = max(deltas, key=lambda t: t[1])
    worst = min(deltas, key=lambda t: t[1])
    return baseline, best, worst


print("=== MLP ===")
base, best, worst = best_and_worst_change(mlp_results)
print(f"baseline: {base['name']} f1={base['best_val_metrics']['f1']:.4f}")
print(f"mejor cambio: {best[0]['name']} (delta F1={best[1]:+.4f}) diffs={hp_diff(base, best[0])}")
print(f"peor cambio: {worst[0]['name']} (delta F1={worst[1]:+.4f}) diffs={hp_diff(base, worst[0])}")

print("\n=== RNN ===")
base_r, best_r, worst_r = best_and_worst_change(rnn_results)
print(f"baseline: {base_r['name']} f1={base_r['best_val_metrics']['f1']:.4f}")
print(f"mejor cambio: {best_r[0]['name']} (delta F1={best_r[1]:+.4f}) diffs={hp_diff(base_r, best_r[0])}")
print(f"peor cambio: {worst_r[0]['name']} (delta F1={worst_r[1]:+.4f}) diffs={hp_diff(base_r, worst_r[0])}")

print("\n=== LSTM ===")
base_l, best_l, worst_l = best_and_worst_change(lstm_results)
print(f"baseline: {base_l['name']} f1={base_l['best_val_metrics']['f1']:.4f}")
print(f"mejor cambio: {best_l[0]['name']} (delta F1={best_l[1]:+.4f}) diffs={hp_diff(base_l, best_l[0])}")
print(f"peor cambio: {worst_l[0]['name']} (delta F1={worst_l[1]:+.4f}) diffs={hp_diff(base_l, worst_l[0])}")

> **Nota:** las respuestas narrativas de esta sección (impacto de hiperparámetros, efecto de la
> regularización, comparación MLP/RNN/LSTM, efecto de la longitud de secuencia y vanishing
> gradient, tipos de reseñas mal clasificadas, y elección de arquitectura para producción) se
> redactan en el **reporte PDF** de este laboratorio, tomando como base exactamente los números
> impresos y las tablas/figuras generadas en este notebook (`summary_df`, `comparison_df`,
> `seqlen_df`, matrices de confusión y ejemplos mal clasificados de la sección 6).

## 7. Guardado de resultados

Persistimos todos los resultados numéricos a `results/results.json` (y las tablas a CSV) para
poder redactar el reporte PDF con los números exactos obtenidos en esta ejecución.

In [ ]:
def strip_history_for_json(res):
    r = dict(res)
    return r

full_dump = {
    "vocab_size": VOCAB_SIZE,
    "max_len_main": MAX_LEN,
    "n_train": len(train_tokens),
    "n_val": len(val_tokens),
    "n_test": len(test_tokens),
    "length_stats": {
        "min": int(lengths.min()), "max": int(lengths.max()),
        "mean": float(lengths.mean()), "median": float(np.median(lengths)),
        "p90": float(np.percentile(lengths, 90)), "p95": float(np.percentile(lengths, 95)),
    },
    "mlp_results": mlp_results,
    "rnn_results": rnn_results,
    "lstm_results": lstm_results,
    "best_mlp": best_mlp_res["name"],
    "best_rnn": best_rnn_res["name"],
    "best_lstm": best_lstm_res["name"],
    "final_test_results": final_test_results,
    "seqlen_results": {f"{a}_{l}": v for (a, l), v in seqlen_results.items()},
    "misclassified_examples": misclassified_examples,
    "best_arch_by_f1": best_arch_by_f1,
}

with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(full_dump, f, indent=2, default=str)

summary_df.to_csv(os.path.join(RESULTS_DIR, "summary_iterations.csv"), index=False)
comparison_df.to_csv(os.path.join(RESULTS_DIR, "comparison.csv"), index=False)
seqlen_df.to_csv(os.path.join(RESULTS_DIR, "seqlen.csv"), index=False)
misclassified_df.to_csv(os.path.join(RESULTS_DIR, "misclassified_examples.csv"), index=False)

print("Resultados guardados en", os.path.abspath(RESULTS_DIR))